# 4 Searching architectures instead of guessing them

A training run now costs about two seconds. That single fact makes hand-tuning
indefensible — not inefficient, *indefensible*, because you can no longer claim you picked
the width by intuition when checking twenty widths would have cost a coffee.

So we stop guessing. This notebook hands the architecture space to a search algorithm and
reads what comes back, honestly, including the parts that are disappointing.

**We switch datasets here, on purpose.** Notebooks 01–03 used EuroSAT, where the point was
that the approach works at all. But EuroSAT's three encoders land within three points of
each other, so a search there is flat on every axis — including the one that is supposed
to matter. DTD (textures) spreads those same encoders across fourteen points and leaves
about twenty points of headroom. Run your search where the result is legible.

In [ ]:
import os
import warnings

# ray propagates `uv run` to its workers, which then expect pyproject.toml in the notebook's
# own directory. It is one level up, so turn that off. Must happen before `import ray`.
os.environ.setdefault("RAY_ENABLE_UV_RUN_RUNTIME_ENV", "0")
warnings.filterwarnings("ignore", message=".*pkg_resources.*")

import sys
import time
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import numpy as np
import torch
from loguru import logger
from torch import nn
from torch.utils.data import DataLoader
from vectormesh import VectorCache
from vectormesh.components import Concatenate2D, GaussianNoise, NeuralNet, Parallel, Serial
from vectormesh.data import Collate, CollateParallel, OneHot

from notebooktester import param

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()

EPOCHS = param(15, test=1)
N_TRIALS = param(24, test=2)
SEEDS = param(5, test=2)

ENCODERS = ["dinov2-small", "resnet-18"]
caches = {
    enc: {split: VectorCache.from_hub(f"pttrn-io/dtd-{enc}", split=split) for split in ("train", "test")}
    for enc in ENCODERS
}

N_CLASSES = caches["dinov2-small"]["train"].features["label"].num_classes
DIM = {enc: caches[enc]["train"].metadata["embed"]["hidden_size"] for enc in ENCODERS}

print(f"DTD: {len(caches['dinov2-small']['train'])} train / {len(caches['dinov2-small']['test'])} test, {N_CLASSES} classes")
print("encoder widths:", DIM)

In [ ]:
onehot = OneHot(num_classes=N_CLASSES, label_col="label", target_col="onehot")
collate = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)


def loaders(encoder: str, batch_size: int = 64):
    return [
        DataLoader(caches[encoder][split].dataset.map(onehot), batch_size=batch_size,
                   shuffle=(split == "train"), collate_fn=collate)
        for split in ("train", "test")
    ]


def run(model, train_dl, test_dl, lr: float = 1e-3, epochs: int = EPOCHS) -> float:
    # the loop from notebook 03, wrapped so a search can call it
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for X, y in train_dl:
            opt.zero_grad()
            loss_fn(model(X), y).backward()
            opt.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in test_dl:
            correct += int((model(X).argmax(1) == y.argmax(1)).sum())
            total += len(y)
    return correct / total

## 4.0 The baseline, before anything else

Notebook 03 said it and it is worth repeating as a rule: **a search without a baseline is
unreadable.** If you do not know what the two-line model scores, you cannot tell whether
the search's best trial is a discovery or a rounding error.

So establish it first, and establish it with a seed spread, because a single run is an
anecdote.

In [ ]:
train_dl, test_dl = loaders("dinov2-small")

scores = []
start = time.time()
for seed in range(SEEDS):
    torch.manual_seed(seed)
    baseline = Serial([NeuralNet(hidden_size=DIM["dinov2-small"], out_size=N_CLASSES)])
    scores.append(run(baseline, train_dl, test_dl))

BASELINE = float(np.mean(scores))
BASELINE_STD = float(np.std(scores))

print(f"seeds: {[f'{s:.4f}' for s in scores]}")
print(f"\nBASELINE  {BASELINE:.4f} +/- {BASELINE_STD:.4f}   ({time.time() - start:.0f}s for {SEEDS} runs)")
print(f"anything inside +/-{2 * BASELINE_STD:.4f} of that is not a result.")

That second number is the one people skip, and it is the one that decides what counts as a
win. Write both down. A trial that beats the mean by less than the spread has not beaten
anything.

## 4.1 The search space

One dictionary describes every model we are willing to consider:

- **`encoder`** — which cache to train on. This is the axis notebook 01 measured at
  fourteen points on this dataset, and it is the reason the search is worth running.
- **`width`** — the size of a hidden layer, where `0` means "no hidden layer", i.e. the
  baseline.
- **`sigma`** — how much Gaussian noise to add to the input vectors during training. Pixel
  augmentation is impossible now (the pixels are gone), so this is the feature-space
  version: it teaches the head not to trust any single dimension too much.
- **`lr`** — sampled log-uniformly, because learning rates matter multiplicatively.

We use **TPE** (`HyperOptSearch`), which models which regions of the space produced good
scores and samples where the expected improvement is highest. A grid search would spend
most of its budget on combinations it already knows are bad. Lesson 4 opens up the
algorithms themselves; here it is one tool, used properly.

In [ ]:
from ray import tune
from ray.tune.search.hyperopt import HyperOptSearch


def trainable(config):
    # ray runs this in its own process, so everything it needs is loaded inside
    encoder = config["encoder"]
    train = VectorCache.from_hub(f"pttrn-io/dtd-{encoder}", split="train")
    test = VectorCache.from_hub(f"pttrn-io/dtd-{encoder}", split="test")
    n_classes = train.features["label"].num_classes
    dim = train.metadata["embed"]["hidden_size"]

    oh = OneHot(num_classes=n_classes, label_col="label", target_col="onehot")
    col = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)
    tr = DataLoader(train.dataset.map(oh), batch_size=64, shuffle=True, collate_fn=col)
    te = DataLoader(test.dataset.map(oh), batch_size=64, shuffle=False, collate_fn=col)

    layers = [GaussianNoise(sigma=config["sigma"])]
    if config["width"]:
        layers += [NeuralNet(hidden_size=dim, out_size=config["width"]),
                   NeuralNet(hidden_size=config["width"], out_size=n_classes)]
    else:
        layers += [NeuralNet(hidden_size=dim, out_size=n_classes)]

    torch.manual_seed(0)
    model = Serial(layers)
    opt = torch.optim.Adam(model.parameters(), lr=config["lr"])
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(config["epochs"]):
        model.train()
        for X, y in tr:
            opt.zero_grad()
            loss_fn(model(X), y).backward()
            opt.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in te:
            correct += int((model(X).argmax(1) == y.argmax(1)).sum())
            total += len(y)
    tune.report({"Accuracy": correct / total})


param_space = {
    "encoder": tune.choice(ENCODERS),
    "width": tune.choice([0, 128, 512]),          # 0 = no hidden layer = the baseline
    "sigma": tune.uniform(0.0, 0.4),
    "lr": tune.loguniform(1e-4, 3e-3),
    "epochs": EPOCHS,
}

In [ ]:
tuner = tune.Tuner(
    trainable,
    param_space=param_space,
    tune_config=tune.TuneConfig(
        metric="Accuracy",
        mode="max",
        search_alg=HyperOptSearch(),
        num_samples=N_TRIALS,
    ),
    run_config=tune.RunConfig(
        storage_path=Path("ray_results").resolve(),
        name="dtd_head_search",
        verbose=0,
    ),
)

start = time.time()
results = tuner.fit()
print(f"{N_TRIALS} trials in {time.time() - start:.0f}s")

## 4.2 Reading the result

The temptation is to read off the winning config and move on. Do the opposite: look at how
accuracy varies along **each axis separately**, which tells you whether the axis mattered
at all.

In [ ]:
df = results.get_dataframe()
df = df[["Accuracy", "config/encoder", "config/width", "config/sigma", "config/lr"]]
df.columns = ["accuracy", "encoder", "width", "sigma", "lr"]

print(df.sort_values("accuracy", ascending=False).head(8).to_string(index=False))

print(f"\n{'encoder':<14} {'trials':>7} {'worst':>8} {'best':>8}")
print("-" * 40)
for enc, group in df.groupby("encoder"):
    print(f"{enc:<14} {len(group):>7} {group['accuracy'].min():>8.4f} {group['accuracy'].max():>8.4f}")

gap = df[df.encoder == "dinov2-small"]["accuracy"].min() - df[df.encoder == "resnet-18"]["accuracy"].max()
print(f"\ngap between the worst dinov2 trial and the best resnet trial: {100 * gap:+.1f} pt")
print(f"\nbaseline (no search, {SEEDS} seeds): {BASELINE:.4f} +/- {BASELINE_STD:.4f}")
print(f"best trial found by the search    : {df['accuracy'].max():.4f}")
print(f"width of the best trial           : {int(df.loc[df['accuracy'].idxmax(), 'width'])}  (0 = the baseline)")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))

for ax, col in zip(axes, ["encoder", "width", "sigma", "lr"]):
    if col in ("encoder", "width"):
        groups = df.groupby(col)["accuracy"]
        ax.scatter(df[col].astype(str), df["accuracy"], alpha=0.7)
        for name, group in groups:
            ax.plot([str(name)], [group.mean()], marker="_", ms=40, color="crimson", mew=2)
    else:
        ax.scatter(df[col], df["accuracy"], alpha=0.7)
        if col == "lr":
            ax.set_xscale("log")
    ax.axhline(BASELINE, color="k", ls="--", lw=1)
    ax.axhspan(BASELINE - BASELINE_STD, BASELINE + BASELINE_STD, color="k", alpha=0.08)
    ax.set_xlabel(col)

axes[0].set_ylabel("test accuracy")
fig.suptitle("one axis at a time; dashed line = the two-line baseline, band = its seed spread")
plt.tight_layout()

Read the four panels left to right and the shape of the answer is unmistakable.

**The encoder panel separates into two bands.** The gap printed above is the whole story:
the *worst* DINOv2 trial in the search still beats the *best* resnet-18 trial, and nothing
done to the head closes it. Every knob in this search was chosen to give the head a chance,
and none of them rescue the weaker representation.

**The other three panels are clouds.** Width, sigma and learning rate scatter their trials
up and down *within* a band without organising them. Inside the good band the visible
variation is largely the seed noise you measured in §4.0 — remember that each trial here is
a single seed.

And look at the `width` column of the leaderboard: the best trials are the ones with
`width = 0`. **The search's answer to "what architecture should I use" was "the one you
already had".** Say that out loud, because it is the result:

> **The leverage was upstream, in the encoder, all along.** The design space between the
> vectors and the loss — the part that feels like "doing machine learning" — is nearly flat
> on a frozen encoder.

### Do not over-generalise this

This holds for **a small head on a strong frozen encoder**. It is not a law about neural
architecture, and two later lessons are counter-examples by construction:

- **Lesson 3 (time series)** has no general-purpose encoder worth freezing. You window the
  series and train end to end, and the architecture is back to being your problem.
- **Lesson 6 (self-supervised)** is about *making* the representation instead of renting
  one. That is precisely the place where architecture decides everything.

The transferable lesson is not "architecture never matters". It is: **find out where the
leverage is on your problem before spending your week on the part that feels most like
engineering.**

## 4.3 The fusion arm

Here is an idea with a good story attached. We have two encoders. DINOv2 is strong, resnet
is weak — but they were trained differently, so perhaps they made *different* mistakes. If
resnet knows anything DINOv2 does not, a model that sees both should beat either alone.

That is a real hypothesis, it is cheap to test, and this is how it is built:
`join` the two caches on `source_idx`, collate two columns instead of one, run a `Parallel`
branch per stream and `Concatenate2D` the results.

**Before you run the next cell, commit to a prediction.** Write it down — sign and rough
size. Does fusing a strong and a weak encoder beat the strong one alone, and by how much?

A prediction you got wrong teaches you more than one you were handed.

In [ ]:
def joined_loaders(batch_size: int = 64):
    out = []
    for split in ("train", "test"):
        joined = caches["dinov2-small"][split].join(caches["resnet-18"][split], on="source_idx")

        # D12: row order happens to match today. Relying on that silently is how you get a
        # model trained on mismatched pairs, so assert it instead of hoping.
        assert (joined["source_idx"][:] == caches["dinov2-small"][split]["source_idx"][:]).all()

        col2 = next(c for c in joined.column_names if c.startswith("embed_"))
        collate2 = CollateParallel(vec1_col="embed", vec2_col=col2,
                                   target_col="onehot", padder=torch.stack)
        out.append(DataLoader(joined.dataset.map(onehot), batch_size=batch_size,
                              shuffle=(split == "train"), collate_fn=collate2))
    return out


fuse_train, fuse_test = joined_loaders()
(X1, X2), y = next(iter(fuse_train))
print(f"two streams per item: {tuple(X1.shape)} and {tuple(X2.shape)}  ->  target {tuple(y.shape)}")


def make_fusion():
    d1, d2 = DIM["dinov2-small"], DIM["resnet-18"]
    return Serial([
        Parallel([nn.Identity(), nn.Identity()]),          # both streams, raw widths
        Concatenate2D(),                                    # (B, 384) + (B, 512) -> (B, 896)
        NeuralNet(hidden_size=d1 + d2, out_size=N_CLASSES),
    ])

Note that `Concatenate2D` takes two *different* widths, 384 and 512. That is worth a
footnote: until `vectormesh` 0.5.0 it could not, and every fusion experiment behind this
course had to squeeze both branches through a common bottleneck first. Keep that in mind
when we get to the result — it turns out to have mattered.

Now run it. **One seed first**, the way you would if you were in a hurry.

In [ ]:
torch.manual_seed(0)
one_seed = run(make_fusion(), fuse_train, fuse_test)

print(f"fusion, seed 0   : {one_seed:.4f}")
print(f"baseline, seed 0 : {BASELINE:.4f}")
print(f"difference       : {100 * (one_seed - BASELINE):+.2f} pt")

Whatever sign that came out, **it is not yet a result.** You have one number from one seed
against a mean of several. §4.0 measured how much this task wobbles between seeds; a
single run cannot see past that.

So do it properly.

In [ ]:
def seed_spread(make_model, train_dl, test_dl, seeds: int = SEEDS) -> np.ndarray:
    out = []
    for seed in range(seeds):
        torch.manual_seed(seed)
        out.append(run(make_model(), train_dl, test_dl))
    return np.array(out)


single = np.array(scores)                                   # the baseline runs from 4.0
fusion = seed_spread(make_fusion, fuse_train, fuse_test)

n_params = lambda m: sum(p.numel() for p in m.parameters())
diff = fusion.mean() - single.mean()
pooled = np.sqrt(fusion.var(ddof=1) / len(fusion) + single.var(ddof=1) / len(single))

print(f"{'setup':<22} {'mean':>8} {'std':>8} {'params':>10}")
print("-" * 52)
print(f"{'baseline (dinov2)':<22} {single.mean():>8.4f} {single.std():>8.4f} "
      f"{n_params(Serial([NeuralNet(hidden_size=DIM['dinov2-small'], out_size=N_CLASSES)])):>10,}")
print(f"{'fusion (both)':<22} {fusion.mean():>8.4f} {fusion.std():>8.4f} {n_params(make_fusion()):>10,}")
print()
print(f"difference      : {100 * diff:+.2f} pt")
print(f"in sigma        : {diff / pooled:+.1f}")
print(f"fusion wins in  : {int((fusion > single.mean()).sum())}/{len(fusion)} seeds")
print(f"variance cost   : {fusion.std() / single.std():.1f}x the baseline's seed spread")

### What to write in the report

Run over enough seeds, this is what the experiment says — and it is neither of the two
tidy answers you were probably expecting:

**Fusion is not better and not worse. It is the same, and it costs more.** The mean moves
by a fraction of a point, well inside one sigma; it wins most seeds but not by an amount
that survives the noise. For that it spends roughly three times the parameters and, more
tellingly, several times the *seed variance* — the fused model is a less predictable thing
to train.

"No measurable difference, at a cost" is a complete result. Write it down and move on. That
is what the two seconds bought you: a real answer to a real question, cheap enough that you
never had to have an opinion about it.

### Two admissions, because they are the actual lesson

**This course got this wrong once already.** Earlier measurements had fusion losing five
setups out of five, and that was very nearly written into these notes as "fusion loses".
It was an artefact: every one of those runs squeezed both branches through a 128-wide
bottleneck first, because `Concatenate2D` could not handle unequal widths. The bottleneck
was throwing away more than the second encoder was adding. Remove it, and the loss goes
away. A finding about *our tooling* was one edit away from being taught as a fact about
fusion.

**And you watched a win evaporate two cells ago.** Seed 0 on its own had fusion ahead by
well over a point — a result you would happily have put in a report. Averaged over several
seeds it shrank to a fraction of that, inside one sigma. Nothing dishonest happened; one
seed was simply not enough to see a difference that small, and the fused model's spread is
several times the baseline's, which makes any single draw from it especially unreliable.

The same thing happened to us while writing this notebook, one level up: a three-seed run
of this experiment reported +0.83 pt and an eight-seed run reported +0.33. Small results
evaporate under more seeds. That is the ordinary case, not the embarrassing exception, and
it is why §4.0 makes you measure the spread before you measure anything else.

### So is fusion good or bad?

Neither, and the question is malformed. What you have measured is that *this* fusion, of
*these* two encoders, on *this* dataset, at *this* training size, is a wash. Change any of
those and the sign can change with it — a genuinely complementary second stream (text
alongside images, a different sensor, a different modality) is a completely different
proposition from a weaker view of the same pixels.

A student who leaves this notebook believing "fusion is bad" has learned something false
just as surely as one who leaves believing "fusion is good". What is worth keeping is the
loop:

> have an idea → state it so it could fail → build it cheaply → measure it against a
> baseline you established **first** → report what happened, including "nice idea, no
> difference"

The cache is what makes that loop cost two seconds. That is the real reason this lesson
started there.

## What you now know

- When a run costs seconds, searching beats guessing — and TPE beats grid search at it.
- Read a search one axis at a time. An axis that does not organise the scatter did not
  matter.
- On a frozen encoder the leverage is upstream: the encoder separates cleanly, the head
  knobs are mostly noise. This is *not* a general law about architecture — see L3 and L6.
- Establish the baseline and its seed spread **before** searching, or you cannot tell a
  win from a wobble.
- A negative or null result, reported with its variance, is a complete result.

## Exercises

Notebook 05 is where you design and run your own search. Two short ones first:

**E1 — how many trials do you actually need?** Rerun the search with `N_TRIALS = 8` and
`N_TRIALS = 48`. Does the best config change? Does the *conclusion* change? Which of those
two questions is the one that matters?

**E2 — take the bottleneck back out.** Rebuild `make_fusion` the old way — project both
branches to 128 before concatenating — and measure it over seeds. You should be able to
reproduce the loss described above. Explain, in one sentence, where the information went.